# 🇰🇭 Fine-tune Khmer LLM — QLoRA on SEA-LION 8B (you2show repos)

QLoRA fine-tuning (4-bit) លើ `you2show/Llama-SEA-LION-v3-8B-IT-bucket` (copy សាធារណៈរបស់អ្នក —
មិនត្រូវ gated login) ដោយប្រើ distillation dataset របស់អ្នក។ រត់លើ **Colab/Kaggle T4 ឥតគិតថ្លៃ**។

**ហេតុអ្វី SEA-LION?** pretrain ជាមួយខ្មែរផ្ទាល់ → base ខ្មែរខ្លាំង, ត្រូវ data តិចជាង។
**រយៈពេល**: ~30-60 នាទី / 3 epochs (subset)។ HF profile៖ https://huggingface.co/you2show

## ជំហានទី ០ — បើក GPU
- **Colab**: Runtime → Change runtime type → **T4 GPU**
- **Kaggle**: Settings ⚙️ → Accelerator → **GPU T4 x2** ឬ **P100**

In [ ]:
# ដំឡើង library (~2-3 នាទី)
!pip install -q -U "transformers>=4.46" "trl>=0.24" peft bitsandbytes accelerate datasets sentencepiece

In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTConfig, SFTTrainer

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0),
          "| VRAM:", round(torch.cuda.get_device_properties(0).total_memory/1e9,1),"GB")
else:
    print("⚠️  គ្មាន GPU — ត្រឡប់ទៅជំហានទី ០ បើក GPU សិន")

## Config — កែត្រង់នេះ

In [ ]:
# model bucket សាធារណៈរបស់អ្នក (មិនត្រូវ gated login)
MODEL_ID = "you2show/Llama-SEA-LION-v3-8B-IT-bucket"
# distillation dataset របស់អ្នក
DATASET_ID = "you2show/GPT-5.5-Gemini-3.1-Pro-Grok-4-Claude-Fable-5-Mythos-5-Qwen-3.7-Max-and-more-Distillation-Dataset"
OUTPUT_DIR = "./sealion-khmer-lora"
MAX_LENGTH = 1024
NUM_EPOCHS = 3
PER_DEVICE_BATCH_SIZE = 2
GRAD_ACCUMULATION = 8      # effective batch = 16
LEARNING_RATE = 2e-4
MAX_TRAIN_EXAMPLES = 500   # None = ទាំងអស់ (ចាប់ផ្តើមតូចដើម្បីតេស្ត pipeline)

# ជ្រើសយកតែ row ភាសាខ្មែរ (dataset នេះលាយច្រើនភាសា → train ខ្មែរឲ្យផ្តោត)
KHMER_ONLY = True         # False = ប្រើគ្រប់ភាសា
KHMER_MIN_CHARS = 5       # យ៉ាងតិចប៉ុន្មានអក្សរខ្មែរ ទើបរាប់ថាជា row ខ្មែរ

# លាយ/បន្ថែម data ខ្មែរ (ការពារ dataset តូច និង catastrophic forgetting)
EXTRA_KHMER_DATASET = "saillab/alpaca-khmer-cleaned"  # "" = កុំបន្ថែម
KEEP_ENGLISH_RATIO = 0.15   # រក្សា English ខ្លះធៀបនឹងខ្មែរ (0 = ខ្មែរសុទ្ធ)

## (ជម្រើស) Login Hugging Face

Repo `-bucket` សាធារណៈ → ជាទូទៅ **មិនត្រូវ login**។ ដោះ comment តែពេល repo ជា private
ឬពេលប្តូរទៅ model gated (ឧ. `aisingapore/…` ដើម)។

In [ ]:
# from huggingface_hub import login; login()

## ជំហានទី ១ — ទាញ dataset + មើលទម្រង់ពិត

In [ ]:
raw_dataset = load_dataset(DATASET_ID)
print(raw_dataset)
SPLIT = "train" if "train" in raw_dataset else list(raw_dataset.keys())[0]
print("\nប្រើ split:", SPLIT)
print("Columns:", raw_dataset[SPLIT].column_names)
print("\n--- ឧទាហរណ៍ទី ០ ---")
print(raw_dataset[SPLIT][0])
POOL = raw_dataset[SPLIT]   # default; ជំហាន ១.៥ អាច filter វា

## ជំហានទី ១.៥ — ជ្រើសយកតែ row ភាសាខ្មែរ (ជម្រើស)

Dataset នេះធំ (~18.5M rows) ហើយលាយច្រើនភាសា។ បើ `KHMER_ONLY = True` cell នេះ scan រក
អក្សរខ្មែរ (Unicode U+1780–U+17FF) ក្នុង `instruction`+`response` ហើយទុកតែ row ខ្មែរ។
បើរកមិនឃើញ row ខ្មែរ វា **ថយក្រោយទៅប្រើគ្រប់ row** ដើម្បីកុំឲ្យ dataset ទទេ។
⏳ scan ពេញ 18.5M អាចចំណាយពេលពីរបីនាទី។

In [ ]:
import re
KHMER_RE = re.compile(r"[\u1780-\u17FF]")

def is_khmer_row(ex):
    txt = f"{ex.get('instruction','')} {ex.get('response','')}"
    return len(KHMER_RE.findall(txt)) >= KHMER_MIN_CHARS

if KHMER_ONLY:
    before = len(POOL)
    try:
        filtered = POOL.filter(is_khmer_row, num_proc=4)
    except Exception:
        filtered = POOL.filter(is_khmer_row)          # num_proc អាចមិនដំណើរការលើ env ខ្លះ
    print(f"row ខ្មែរ: {len(filtered):,} / {before:,}")
    if len(filtered) > 0:
        POOL = filtered
    else:
        print("⚠️  រកមិនឃើញ row ខ្មែរ — ប្រើគ្រប់ភាសាវិញ (កែ KHMER_MIN_CHARS ឬ KHMER_ONLY=False)")
else:
    print(f"KHMER_ONLY=False → ប្រើគ្រប់ {len(POOL):,} row")

## ជំហានទី ២ — tokenizer + model (4-bit)

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,   # T4/P100 មិនគាំទ្រ bf16 ពេញលេញ
    bnb_4bit_use_double_quant=True,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb_config, device_map="auto",
)
model.config.use_cache = False
print("Model loaded ✅")

## ជំហានទី ៣ — Format data (auto-detect schema)

Dataset របស់អ្នកមាន column `instruction` + `response` (ព្រម `source`/`category` ជា metadata)។
ចំណាំ៖ `instruction` ខ្លះផ្ទុក structure `{'messages': [...]}` ជា string — function ខាងក្រោម
**parse វាចេញ** ដោយស្វ័យប្រវត្តិ (json រួច Python-dict) រួចភ្ជាប់ `response` ជាចម្លើយ។
វាក៏ស្គាល់ `messages`, ShareGPT `conversations`, `prompt/completion`, `question/answer`, `text`
ដែរ ដូច្នេះដំណើរការទោះ dataset ប្រើ schema ណា។

Cell នេះក៏ **លាយ data** ផង៖ ខ្មែរពី distillation + `EXTRA_KHMER_DATASET` (បរិមាណបន្ថែម) + English ខ្លះ (`KEEP_ENGLISH_RATIO`, ការពារ forgetting)។

In [ ]:
import json, ast

ROLE_MAP = {"human":"user","user":"user","gpt":"assistant","assistant":"assistant",
            "system":"system","bot":"assistant","model":"assistant"}

def _parse_messages(s):
    """បើ field ជា string ដែលផ្ទុក {'messages': [...]} (ដូច dataset នេះ) → parse យក។"""
    if not isinstance(s, str):
        return None
    t = s.strip()
    if not (t.startswith("{") or t.startswith("[")):
        return None
    for parser in (json.loads, ast.literal_eval):   # json ជាមុន, រួច Python-dict (single quotes)
        try:
            obj = parser(t)
        except Exception:
            continue
        if isinstance(obj, dict) and isinstance(obj.get("messages"), list):
            return obj["messages"]
        if isinstance(obj, list) and obj and isinstance(obj[0], dict) and "role" in obj[0]:
            return obj
    return None

def to_messages(ex):
    # 1) chat messages ស្រាប់
    if ex.get("messages"):
        return ex["messages"]
    # 2) ShareGPT conversations
    if ex.get("conversations"):
        return [{"role": ROLE_MAP.get(str(t.get("from","user")).lower(),"user"),
                 "content": t.get("value") or t.get("content") or ""} for t in ex["conversations"]]
    # 3) instruction/prompt + response/output  (dataset នេះ៖ instruction + response)
    for uk in ("instruction","prompt","question","input_text","query"):
        if ex.get(uk) in (None, ""):
            continue
        for ak in ("response","output","completion","answer","output_text","chosen"):
            if ex.get(ak) is None:
                continue
            answer = {"role": "assistant", "content": str(ex[ak]).strip()}
            inner = _parse_messages(ex[uk])         # instruction អាចជា {'messages': [...]}
            if inner is not None:
                return list(inner) + [answer]
            user = str(ex[uk]).strip()
            extra = str(ex.get("input") or "").strip()
            if extra and uk != "input":
                user = f"{user}\n\n{extra}"
            return [{"role": "user", "content": user}, answer]
    return None

def format_example(ex):
    msgs = to_messages(ex)
    if not msgs:
        return {"text": str(ex["text"]) if ex.get("text") else ""}
    return {"text": tokenizer.apply_chat_template(msgs, tokenize=False)}

def cap(ds):
    return ds.select(range(min(MAX_TRAIN_EXAMPLES, len(ds)))) if MAX_TRAIN_EXAMPLES else ds

def format_dataset(ds):
    return ds.map(format_example, remove_columns=ds.column_names)\
             .filter(lambda r: len(r["text"].strip()) > 0)

# ---- ប្រមូល source data ----------------------------------------------------
sources = [("distillation-khmer", POOL)]          # ខ្មែរ ពី distillation (ជំហាន ១.៥)

if EXTRA_KHMER_DATASET:                            # dataset ខ្មែរ dedicated (បរិមាណបន្ថែម)
    try:
        _x = load_dataset(EXTRA_KHMER_DATASET)
        _xs = "train" if "train" in _x else list(_x.keys())[0]
        sources.append((EXTRA_KHMER_DATASET, _x[_xs]))
        print(f"បន្ថែម {len(_x[_xs]):,} row ពី {EXTRA_KHMER_DATASET}")
    except Exception as e:
        print(f"⚠️ ទាញ {EXTRA_KHMER_DATASET} មិនបាន — រំលង: {e}")

if KEEP_ENGLISH_RATIO > 0 and KHMER_ONLY:          # រក្សា English ខ្លះ (ការពារ forgetting)
    try:
        _scan = raw_dataset[SPLIT].select(range(min(len(raw_dataset[SPLIT]), 200_000)))
        _eng = _scan.filter(lambda ex: not is_khmer_row(ex))
        _n = int(len(POOL) * KEEP_ENGLISH_RATIO)
        if _n > 0 and len(_eng) > 0:
            _eng = _eng.shuffle(seed=42).select(range(min(_n, len(_eng))))
            sources.append(("english-mix", _eng))
            print(f"រក្សា {len(_eng):,} row English")
    except Exception as e:
        print(f"⚠️ English mix រំលង: {e}")

# ---- format each → {"text"} → concat → shuffle → cap ----------------------
from datasets import concatenate_datasets
parts = []
for name, ds in sources:
    fp = format_dataset(cap(ds))
    print(f"  {name}: {len(fp):,} ត្រឹមត្រូវ")
    if len(fp):
        parts.append(fp)

assert parts, "❌ គ្មាន data — មើល column នៅ cell ១ រួចបន្ថែមក្នុង to_messages"
formatted_dataset = concatenate_datasets(parts).shuffle(seed=42)
if MAX_TRAIN_EXAMPLES:
    formatted_dataset = formatted_dataset.select(range(min(MAX_TRAIN_EXAMPLES, len(formatted_dataset))))

print("\nសរុប train:", len(formatted_dataset))
print("\n--- ឧទាហរណ៍ ---\n", formatted_dataset[0]["text"][:800])

## ជំហានទី ៤ — LoRA

In [ ]:
model = prepare_model_for_kbit_training(model)
lora_config = LoraConfig(
    r=16, lora_alpha=32,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## ជំហានទី ៥ — Trainer

⚠️ `trl` ប្តូរ API ញឹកញាប់។ បើ `TypeError` អំពី `tokenizer=`/`dataset_text_field=`/`max_seq_length=`
នោះ version ថ្មីប្តូរឈ្មោះ។ កូដនេះប្រើ syntax បច្ចុប្បន្ន (SFTConfig + `processing_class`)។

In [ ]:
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    dataset_text_field="text",
    max_length=MAX_LENGTH,
    packing=False,
    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUMULATION,
    gradient_checkpointing=True,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    fp16=True,
    optim="paged_adamw_8bit",
    logging_steps=10,
    save_strategy="epoch",
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    report_to="none",
)
trainer = SFTTrainer(
    model=model, args=training_args,
    train_dataset=formatted_dataset, processing_class=tokenizer,
)

## ជំហានទី ៦ — Train 🚀

In [ ]:
trainer.train()

## ជំហានទី ៧ — រក្សាទុក adapter

In [ ]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("រក្សាទុករួច:", OUTPUT_DIR)
# រក្សាទុកអចិន្ត្រៃយ៍ទៅ HF (កុំឲ្យ Colab លុប)៖
# model.push_to_hub("you2show/sealion-khmer-lora")

## ជំហានទី ៨ — សាកល្បង

In [ ]:
model.config.use_cache = True
model.eval()
msgs=[{"role":"user","content":"សូមណែនាំរបៀបធ្វើម្ហូបខ្មែរសាមញ្ញមួយមុខ"}]
inputs=tokenizer.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True, return_tensors="pt").to(model.device)
with torch.no_grad():
    out=model.generate(inputs, max_new_tokens=256, do_sample=True, temperature=0.7)
print(tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True))

## ជំហានទី ៩ — ប្រើក្នុង A2I

Training រក្សាទុកតែ **LoRA adapter** (តូច)។  

**ផ្លូវងាយបំផុត — vLLM serve adapter ផ្ទាល់ (មិនចាំបាច់ merge):**
vLLM serve LoRA adapter ដោយផ្ទាល់ ដូច្នេះមិនចាំបាច់ merge→GGUF ទេ (ជៀសបញ្ហា RAM 8B)៖

```bash
vllm serve you2show/Llama-SEA-LION-v3-8B-IT-bucket \
    --enable-lora --lora-modules khmer=./sealion-khmer-lora
# រួច A2I web → Settings → AI providers → http://HOST:8000/v1 (model: khmer)
```

**ឬ** បំប្លែងជា GGUF សម្រាប់ A2I Core / llama.cpp៖
ដើម្បីបំប្លែងជា GGUF ត្រូវ **reload base fp16**
សិន (កុំ `merge_and_unload` លើ 4-bit ដែលទើប train), រួច merge → convert៖

```python
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM
base = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16)
merged = PeftModel.from_pretrained(base, OUTPUT_DIR).merge_and_unload()
merged.save_pretrained("sealion-khmer-merged"); tokenizer.save_pretrained("sealion-khmer-merged")

!git clone https://github.com/ggerganov/llama.cpp && pip install -q -r llama.cpp/requirements.txt
!python llama.cpp/convert_hf_to_gguf.py sealion-khmer-merged --outfile sealion-khmer.gguf --outtype q8_0
!./llama.cpp/llama-quantize sealion-khmer.gguf sealion-khmer-q4_k_m.gguf q4_k_m
```

⚠️ 8B fp16 merge ត្រូវ ~16GB RAM — លើសពី Colab free។ Merge លើម៉ាស៊ីនធំ (A100/local),
ឬ push adapter ទៅ HF មុន។ បន្ទាប់មក ដាក់ GGUF នៅ `a2i-core/models/model.gguf` → `./run.sh`
(មើល `a2i-train/README.md`)។ ឬ serve GGUF លើ Colab តាម `a2i-core/serve_gguf_colab.ipynb`
រួចភ្ជាប់ A2I web។